On commence par importer les bibliothèques nécessaires. Attention : NE PAS EFFACER !

In [ ]:
%run Import_bibliotheques.ipynb

%run Arduino.ipynb

connexion_arduino()

In [ ]:
# Choix du nombre de mesures à effectuer :
while True :
    try :
        nb_mesures = int(input("Combien de mesures souhaitez-vous effectuer ? "))
        if nb_mesures > 0 :
            break
        print("Veuillez entrer un nombre strictement positif.")
    except ValueError :
        print("Veuillez entrer un nombre entier.")
print(f"{nb_mesures} mesures seront effectuées.")

In [ ]:

if 'ser' not in globals() or not ser.is_open :
    connexion_arduino()
reset_arduino()    # Réinitialisation de l'Arduino et purge du tampon

# Liste pour stocker les tensions mesurées
liste_donnees = []
lectures_vides = 0
premiers_rejets = 10    # on écarte les 10 premières lectures :
# juste après le reset de la carte, l'entrée analogique et le buffer
# série USB ne sont pas encore stabilisés (valeurs aberrantes).

while len(liste_donnees) < nb_mesures :
    try:
        ligne = ser.readline()    # Lire une ligne de données depuis le port série
        parse = parse_ligne(ligne)    # format "temps_us,tension" ; None si ligne invalide
        if parse is None:
            if ligne:
                lectures_vides = 0    # ligne reçue mais invalide : on continue
            else:
                lectures_vides += 1    # aucune donnée reçue
                if lectures_vides > 10 :
                    print("Plus de données reçues de l'Arduino : arrêt.")
                    break
            continue
        if premiers_rejets > 0:    # premières lignes instables : non comptées
            premiers_rejets -= 1
            continue
        liste_donnees.append(parse[1])    # on ne garde que la tension
        if len(liste_donnees) % 1000 == 0 :
            # Sauvegarde de secours : on ne perd rien si le kernel est arrêté
            with open("statistiques.csv", 'w') as fichier :
                fichier.write("tension\n" + "\n".join(map(str, liste_donnees)) + "\n")
            print(f"{len(liste_donnees)} mesures enregistrées...")
    except KeyboardInterrupt:
        # Interrompre la boucle avec le bouton « stop » du notebook
        break

# Sauvegarde finale des données
with open("statistiques.csv", 'w') as fichier :
    fichier.write("tension\n" + "\n".join(map(str, liste_donnees)) + "\n")
print(f"Acquisition terminée : {len(liste_donnees)} mesures sauvegardées dans statistiques.csv.")

In [ ]:
if len(liste_donnees) == 0 :
    raise Exception("Aucune mesure enregistrée : relancer la cellule d'acquisition.")
donnees = array(liste_donnees)
print(f"nombre de mesures : {len(donnees)}")

Calculs statistiques (moyenne, écart-type et incertitude-type) sur une série de mesures x :

In [ ]:
variable = donnees
moy = mean(variable)
ecartType = std(variable, ddof=1)  # ddof=1 permet de calculer l'écart-type expérimental Sn-1
incertitudeType = ecartType / sqrt(len(variable))

Affichage des valeurs statistiques :

In [ ]:
resultat = f"moyenne : {moy:.5} ; incertitude-type : {incertitudeType:.2}"    # crée une ligne de texte résumant les résultats
print(resultat)    # affiche le résultat

On trace l'histogramme :

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=variable, nbinsx=100, name=resultat))

# Cosmétique du graphique :
#fig.update_yaxes(scaleanchor="x", scaleratio=1)    # à décommenter pour un repère orthonormé
fig.update_xaxes(showgrid=True)    # grille verticale (par défaut)
fig.update_yaxes(showgrid=True)    # grille horizontale (par défaut)
fig.update_layout(
    showlegend=True,    # légende du graphique,
    title="",    # mettre le titre désiré entre les guillemets,
    xaxis_title="x",    # nom de l'axe des abscisses,
    yaxis_title="Nombre de tirages",    # nom de l'axe des ordonnées,
    height=600)    # hauteur du graphique en pixels ; ex. : 450, 700...